# Particle swarm optimization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Particles move by inertia, memory of their own best, and attraction to the swarm best
Particle swarm optimization turns search into social motion. Each particle carries a velocity, remembers where it personally did well, and is pulled toward the best position discovered by the swarm.

In [ ]:
# 1) One velocity update shows the three PSO forces
x=np.array([0.,4.]); v=np.array([0.,0.]); pbest=x.copy(); gbest=np.array([0.,0.]); w=0.5; c1=1.; c2=1.; r1=np.array([0.2,0.2]); r2=np.array([0.3,0.3])
newv=w*v+c1*r1*(pbest-x)+c2*r2*(gbest-x); newx=x+newv
plt.figure(figsize=(4,4)); plt.arrow(x[0],x[1],newv[0],newv[1],head_width=.08,length_includes_head=True); plt.scatter([x[0],newx[0],gbest[0]],[x[1],newx[1],gbest[1]],s=80); plt.axis('equal'); plt.title('social pull moves the particle downward')
assert np.allclose(newv,[0.,-1.2]) and np.allclose(newx,[0.,2.8])

In [ ]:
# 2) Inertia carries momentum when memory pulls are zero
v=np.array([1.,-0.5]); wvals=np.array([0.2,0.7,1.0]); speeds=np.array([np.linalg.norm(w*v) for w in wvals])
plt.figure(figsize=(6,3)); plt.bar([str(w) for w in wvals], speeds); plt.xlabel('inertia w'); plt.ylabel('speed after inertia'); plt.title('larger inertia preserves motion')
assert speeds[2] > speeds[1] > speeds[0]

In [ ]:
# 3) Cognitive versus social weights change the direction
x=np.array([2.,2.]); pbest=np.array([2.,0.]); gbest=np.array([0.,2.]); cases=[(2,0.5),(0.5,2)]; steps=[]
for c1,c2 in cases: steps.append(c1*0.5*(pbest-x)+c2*0.5*(gbest-x))
steps=np.array(steps); plt.figure(figsize=(4,4)); plt.scatter([x[0]],[x[1]],c='k'); plt.arrow(2,2,steps[0,0],steps[0,1],color='tab:blue',head_width=.07,length_includes_head=True,label='cognitive-heavy'); plt.arrow(2,2,steps[1,0],steps[1,1],color='tab:orange',head_width=.07,length_includes_head=True,label='social-heavy'); plt.legend(); plt.axis('equal'); plt.title('weights decide whose memory matters')
assert abs(steps[0,1]) > abs(steps[0,0]) and abs(steps[1,0]) > abs(steps[1,1])

In [ ]:
# 4) A small swarm converges in 2D
rng=np.random.default_rng(10); target=np.array([1.,-1.]); X=rng.uniform(-4,4,(25,2)); V=np.zeros_like(X); P=X.copy(); Pval=np.sum((P-target)**2,axis=1); G=P[np.argmin(Pval)].copy(); hist=[]; start=X.copy()
for _ in range(30):
    r1=rng.random(X.shape); r2=rng.random(X.shape); V=0.6*V+1.4*r1*(P-X)+1.4*r2*(G-X); X=X+V; val=np.sum((X-target)**2,axis=1); better=val<Pval; P[better]=X[better]; Pval[better]=val[better]; G=P[np.argmin(Pval)].copy(); hist.append(Pval.min())
plt.figure(figsize=(4,4)); plt.scatter(start[:,0],start[:,1],alpha=.25,label='start'); plt.scatter(X[:,0],X[:,1],alpha=.8,label='final'); plt.scatter([target[0]],[target[1]],c='r',marker='*',s=120); plt.legend(); plt.axis('equal'); plt.title('swarm contracts near the optimum')
assert hist[-1] < hist[0] and np.linalg.norm(G-target) < 0.1

In [ ]:
# 5) Best fitness history shows rapid collective learning
plt.figure(figsize=(6,3)); plt.semilogy(hist); plt.xlabel('iteration'); plt.ylabel('global-best objective'); plt.title('shared memory makes progress monotone')
assert all(hist[i+1] <= hist[i] + 1e-12 for i in range(len(hist)-1))